# GPU operations: Databricks data starter

**Synthetic data only. This is a prerecorded replay with scheduled recovery.
Recovery in these readings does not demonstrate that an agent repaired a fault.**

This notebook creates four Delta tables for your website and model teammates:
`inventory`, `telemetry`, `workloads`, and `cooling`. It generates 24 simulated
GPUs across NVIDIA, AMD, and Intel labels. Device limits are demo settings,
not specifications for real hardware.

## Import and run

1. In your Databricks workspace, use **Workspace → Import** (or **Import** in
   the target folder menu), choose **File**, and select this `.ipynb` file.
2. Open the imported notebook and select **Serverless** compute.
3. Run the first code cell to see accessible catalogs. `CATALOG` defaults to
   your current catalog; if necessary, replace it with a catalog where you
   may create a schema. Leave `SCHEMA` and `SEED` unchanged initially.
4. Click **Run all**. No package installation, API key, file upload or external
   data download is needed after importing the notebook.
5. Confirm the final counts and the three example query results. Your four
   tables will also appear in **Catalog Explorer** under the printed path.

Re-running with identical data leaves existing tables unchanged. If a table
has different schema or data, this notebook stops before writing any tables.
Choose a **new SCHEMA name**, such as `gpu_ops_hackathon_v2`, to load a new seed
or changed generator. It never overwrites existing table data.

Evaluation labels are generated in a temporary local directory and deleted.
They are **never uploaded to Delta tables or displayed**. Give the agent access
only to the four evidence tables through bounded queries, not to this notebook
or the generator source. Replay queries must filter to the selected `run_id`
and `event_ts <= as_of_ts` so the agent cannot see future samples.

If creation fails with a permission error, select a writable catalog/schema
or ask the workspace owner to grant the required access. Do not delete existing
data to resolve it.


In [ ]:
# First inspect the catalogs available to your account.
display(spark.sql("SHOW CATALOGS"))

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "gpu_ops_hackathon"
SEED = 42

import re

for label, identifier in (("CATALOG", CATALOG), ("SCHEMA", SCHEMA)):
    if not isinstance(identifier, str) or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"{label} must contain only letters, digits and underscores and must start with a letter or underscore.")
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")
print(f"Evidence table location: {CATALOG}.{SCHEMA}")


## Reproducible fixture generator

The following cell embeds `generate.py` so this notebook can run on its own.
It defines functions without running its command-line entry point.


In [ ]:
#!/usr/bin/env python3
"""Create synthetic, reproducible GPU telemetry for a Databricks demo.

Public API: generate_dataset(output_dir, seed=42, scenario="all", overwrite=False).
Only evidence/*.csv should be exposed to a model or an agent. The evaluation/
directory contains scenario labels and must remain outside their data access.

This is a prerecorded replay: recovery happens on a fixed schedule. It is not
proof that an agent action repaired anything. All device limits are demo values,
not manufacturer specifications. No third-party packages are required.
"""

from __future__ import annotations

import argparse
import csv
import json
import math
import os
from pathlib import Path
import random
import shutil
import tempfile
from datetime import datetime, timedelta, timezone
from uuid import NAMESPACE_URL, uuid5


SCENARIOS = ("cooling_fault", "healthy_load", "telemetry_gap", "isolated_error")
START_TIME = datetime(2026, 11, 14, 9, 0, tzinfo=timezone.utc)
SAMPLE_COUNT = 180
INTERVAL_SECONDS = 5
SCHEMA_VERSION = "1.0"
GENERATOR_ID = "gpu-ops-data"

SCHEMAS = {
    "inventory": (
        "gpu_id", "vendor", "model", "rack_id", "server_id", "cooling_zone_id",
        "power_limit_w", "temperature_limit_c",
    ),
    "telemetry": (
        "run_id", "event_ts", "gpu_id", "power_w", "gpu_util_pct", "memory_used_gb",
        "memory_total_gb", "temp_c", "clock_mhz", "throttled",
        "ecc_uncorrected_total", "heartbeat_ok",
    ),
    "workloads": (
        "run_id", "event_ts", "gpu_id", "job_id", "job_state", "progress_units", "priority",
    ),
    "cooling": (
        "run_id", "event_ts", "cooling_zone_id", "inlet_temp_c", "flow_pct",
    ),
}
NULLABLE_METRICS = SCHEMAS["telemetry"][3:-1]


def timestamp(sample_index: int) -> str:
    """Return the timestamp for a sample, including an exclusive end boundary."""
    return (START_TIME + timedelta(seconds=sample_index * INTERVAL_SECONDS)).isoformat().replace("+00:00", "Z")


def build_inventory() -> list[dict]:
    """Return 24 synthetic GPUs: eight per vendor across three mixed-vendor racks."""
    profiles = {
        "NVIDIA": (450.0, 90.0),
        "AMD": (500.0, 88.0),
        "Intel": (350.0, 85.0),
    }
    inventory = []
    for rack, vendors in (("a", ("NVIDIA", "AMD")), ("b", ("AMD", "Intel")), ("c", ("Intel", "NVIDIA"))):
        for server_number, vendor in enumerate(vendors, start=1):
            for _ in range(4):
                power_limit, temperature_limit = profiles[vendor]
                inventory.append({
                    "gpu_id": f"gpu-{len(inventory) + 1:03d}",
                    "vendor": vendor,
                    "model": f"Synthetic-{vendor}",
                    "rack_id": f"rack-{rack}",
                    "server_id": f"server-{rack}{server_number}",
                    "cooling_zone_id": f"zone-{rack}",
                    "power_limit_w": power_limit,
                    "temperature_limit_c": temperature_limit,
                })
    return inventory


def _cooling_factor(index: int) -> float:
    if 60 <= index < 120:
        return min(1.0, (index - 59) / 12)
    if 120 <= index < 132:
        return max(0.0, 1 - (index - 119) / 12)
    return 0.0


def _csv_row(row: dict) -> dict:
    return {key: (str(value).lower() if isinstance(value, bool) else value) for key, value in row.items()}


def _write_run(writers: dict, inventory: list[dict], seed: int, scenario: str) -> dict:
    run_id = str(uuid5(NAMESPACE_URL, f"{GENERATOR_ID}/{SCHEMA_VERSION}/{seed}/{scenario}"))
    # The same seeded background makes scenario comparisons reproducible.
    rng = random.Random(seed)
    progress = {gpu["gpu_id"]: 0 for gpu in inventory}
    for index in range(SAMPLE_COUNT):
        event_ts = timestamp(index)
        phase_active = 60 <= index < 120
        cooling_factor = _cooling_factor(index) if scenario == "cooling_fault" else 0.0
        for zone in ("zone-a", "zone-b", "zone-c"):
            factor = cooling_factor if zone == "zone-b" else 0.0
            writers["cooling"].writerow({
                "run_id": run_id, "event_ts": event_ts, "cooling_zone_id": zone,
                "inlet_temp_c": round(21 + 17 * factor + rng.uniform(-0.3, 0.3), 3),
                "flow_pct": round(100 - 80 * factor + rng.uniform(-0.5, 0), 3),
            })
        for gpu_index, gpu in enumerate(inventory):
            gpu_id = gpu["gpu_id"]
            idle = gpu_id == "gpu-024"
            util = max(0, min(100, 65 + 13 * math.sin(index / 13 + gpu_index) + rng.uniform(-2, 2)))
            power_limit = gpu["power_limit_w"]
            power = power_limit * (0.12 + 0.67 * util / 100)
            temperature = 42 + 0.055 * power + rng.uniform(-1.2, 1.2)
            clock = 1500.0
            memory_total = 64.0
            memory_used = 28 + gpu_index % 5 + rng.uniform(-0.2, 0.2)
            throttled = False
            ecc = 0
            heartbeat = True
            stalled = False
            if idle:
                util = rng.uniform(0, 2)
                power = power_limit * 0.08
                temperature = 35 + rng.uniform(-1, 1)
                clock = 300.0
                memory_used = 0.5
            elif scenario == "healthy_load" and phase_active:
                util = 94 + rng.uniform(-2, 2)
                power = power_limit * (0.86 + rng.uniform(0, 0.03))
                temperature = gpu["temperature_limit_c"] - 10 + rng.uniform(-1, 1)
                memory_used = 48 + rng.uniform(-0.2, 0.2)
            elif scenario == "cooling_fault" and gpu["rack_id"] == "rack-b" and cooling_factor:
                temperature += (gpu["temperature_limit_c"] + 5 - temperature) * cooling_factor
                throttled = temperature >= gpu["temperature_limit_c"]
                if throttled:
                    clock = 825.0
                    power *= 0.78
            elif scenario == "isolated_error" and gpu_id == "gpu-001":
                # These are cumulative lifetime counters. Recovery does not reset them.
                ecc = int(index >= 60) + int(index >= 90)
                if phase_active:
                    util = 2.0
                    power = power_limit * 0.18
                    temperature = 47.0
                    clock = 500.0
                    stalled = True
            if scenario == "telemetry_gap" and gpu_id == "gpu-017" and phase_active:
                heartbeat = False

            if not idle and not stalled:
                progress[gpu_id] += max(1, int(util * clock / 5000))
            # Workload progress is an independent scheduler/application signal,
            # so it can remain available while the GPU collector is unreachable.
            writers["workloads"].writerow({
                "run_id": run_id, "event_ts": event_ts, "gpu_id": gpu_id,
                "job_id": None if idle else f"job-{gpu_id}",
                "job_state": "idle" if idle else "running",
                "progress_units": progress[gpu_id],
                "priority": "high" if gpu_index % 6 == 0 else "normal",
            })
            row = {
                "run_id": run_id, "event_ts": event_ts, "gpu_id": gpu_id,
                "power_w": round(power, 3), "gpu_util_pct": round(util, 3),
                "memory_used_gb": round(memory_used, 3), "memory_total_gb": memory_total,
                "temp_c": round(temperature, 3), "clock_mhz": clock,
                "throttled": throttled, "ecc_uncorrected_total": ecc,
                "heartbeat_ok": heartbeat,
            }
            if not heartbeat:
                for name in NULLABLE_METRICS:
                    row[name] = None
            writers["telemetry"].writerow(_csv_row(row))

    expected = {
        "cooling_fault": {"cause": "shared_cooling_degradation", "affected_gpu_ids": [g["gpu_id"] for g in inventory if g["rack_id"] == "rack-b"], "cooling_zone_id": "zone-b"},
        "healthy_load": {"cause": "healthy_workload_increase", "affected_gpu_ids": [], "should_raise_fault": False},
        "telemetry_gap": {"cause": "collector_unreachable", "affected_gpu_ids": ["gpu-017"], "hardware_failure_established": False},
        "isolated_error": {"cause": "isolated_gpu_error_with_stalled_workload", "affected_gpu_ids": ["gpu-001"]},
    }[scenario]
    return {
        "run_id": run_id, "scenario": scenario, "expected": expected,
        "baseline_start": timestamp(0), "event_start": timestamp(60),
        "scheduled_recovery_start": timestamp(120), "fully_recovered_by": timestamp(132),
        "end_exclusive": timestamp(SAMPLE_COUNT), "idle_negative_control_gpu": "gpu-024",
    }


def _check_target(target: Path, overwrite: bool) -> None:
    if target.is_symlink():
        raise ValueError("Refusing to replace a symbolic-link output directory.")
    if not target.exists():
        return
    if not overwrite:
        raise FileExistsError(f"Output already exists: {target}. Choose another path or use --overwrite.")
    marker = target / "evaluation" / "generation.json"
    try:
        metadata = json.loads(marker.read_text())
    except (OSError, ValueError) as exc:
        raise ValueError("--overwrite only replaces a dataset previously made by this generator.") from exc
    if metadata.get("generator") != GENERATOR_ID or set(p.name for p in target.iterdir()) != {"evidence", "evaluation"}:
        raise ValueError("Refusing to overwrite an unrecognised output directory or extra files.")
    for directory, expected_names in (
        (target / "evidence", {f"{table}.csv" for table in SCHEMAS}),
        (target / "evaluation", {"generation.json", "ground_truth.json"}),
    ):
        if directory.is_symlink() or not directory.is_dir() or {p.name for p in directory.iterdir()} != expected_names:
            raise ValueError("Refusing to overwrite a modified dataset directory.")
        if any(path.is_symlink() or not path.is_file() for path in directory.iterdir()):
            raise ValueError("Refusing to overwrite symlinks or nested directories in a dataset.")


def generate_dataset(output_dir: str | Path, seed: int = 42, scenario: str = "all", overwrite: bool = False) -> dict:
    """Write CSV evidence and private evaluation labels; return generation metadata.

    Writes into a staging directory before swapping the completed dataset into
    place. --overwrite is restricted to recognised generated directories.
    """
    if scenario != "all" and scenario not in SCENARIOS:
        raise ValueError(f"Unknown scenario: {scenario}")
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise TypeError("seed must be an integer")
    target = Path(output_dir).expanduser().absolute()
    _check_target(target, overwrite)
    target.parent.mkdir(parents=True, exist_ok=True)
    stage_root = Path(tempfile.mkdtemp(prefix=f".{target.name}-staging-", dir=target.parent))
    staged = stage_root / "dataset"
    backup = stage_root / "previous"
    preserve_backup = False
    try:
        (staged / "evidence").mkdir(parents=True)
        (staged / "evaluation").mkdir()
        inventory = build_inventory()
        selected = SCENARIOS if scenario == "all" else (scenario,)
        handles = {}
        writers = {}
        try:
            for table, columns in SCHEMAS.items():
                handles[table] = (staged / "evidence" / f"{table}.csv").open("w", newline="", encoding="utf-8")
                writers[table] = csv.DictWriter(handles[table], fieldnames=columns, lineterminator="\n")
                writers[table].writeheader()
            writers["inventory"].writerows(inventory)
            labels = [_write_run(writers, inventory, seed, name) for name in selected]
        finally:
            for handle in handles.values():
                handle.close()
        metadata = {
            "generator": GENERATOR_ID, "schema_version": SCHEMA_VERSION,
            "seed": seed, "scenario_selection": scenario, "run_count": len(selected),
            "samples_per_run": SAMPLE_COUNT, "interval_seconds": INTERVAL_SECONDS,
            "start_time": timestamp(0), "end_exclusive": timestamp(SAMPLE_COUNT),
            "row_counts": {"inventory": 24, "telemetry": 24 * SAMPLE_COUNT * len(selected), "workloads": 24 * SAMPLE_COUNT * len(selected), "cooling": 3 * SAMPLE_COUNT * len(selected)},
            "synthetic": True,
            "note": "Prerecorded replay. Recovery is scheduled, not caused by agent action. Limits are demo settings, not manufacturer specifications.",
        }
        for name, value in (("generation.json", metadata), ("ground_truth.json", {"schema_version": SCHEMA_VERSION, "runs": labels})):
            (staged / "evaluation" / name).write_text(json.dumps(value, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        # Recheck immediately before mutation; never silently replace a new target.
        _check_target(target, overwrite)
        if target.exists():
            os.replace(target, backup)
        try:
            os.replace(staged, target)
        except BaseException:
            if backup.exists():
                try:
                    os.replace(backup, target)
                except OSError as restore_error:
                    preserve_backup = True
                    raise RuntimeError(f"Could not restore the previous dataset; preserved it at {backup}") from restore_error
            raise
        return metadata
    finally:
        if not preserve_backup:
            shutil.rmtree(stage_root)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output", default="data", help="New dataset directory (default: data).")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--scenario", choices=("all",) + SCENARIOS, default="all")
    parser.add_argument("--overwrite", action="store_true", help="Replace a recognised generated dataset.")
    args = parser.parse_args()
    try:
        metadata = generate_dataset(args.output, args.seed, args.scenario, args.overwrite)
    except (OSError, ValueError, TypeError) as exc:
        parser.exit(1, f"Error: {exc}\n")
    print(json.dumps({"output": str(Path(args.output).absolute()), **metadata}, indent=2))


In [ ]:
# Keep every file, including evaluation labels, on the driver in /tmp.
# Copy only the four CSV evidence datasets into memory, then delete all files.
with tempfile.TemporaryDirectory(prefix="gpu-ops-notebook-", dir="/tmp") as temp_dir:
    fixture_dir = Path(temp_dir) / "dataset"
    generation_info = generate_dataset(fixture_dir, seed=SEED, scenario="all")
    evidence_rows = {}
    for table_name, field_names in SCHEMAS.items():
        with (fixture_dir / "evidence" / f"{table_name}.csv").open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            if tuple(reader.fieldnames) != field_names:
                raise ValueError(f"Unexpected CSV schema for {table_name}")
            evidence_rows[table_name] = list(reader)

expected_counts = {"inventory": 24, "telemetry": 17280, "workloads": 17280, "cooling": 2160}
for table_name, expected in expected_counts.items():
    if len(evidence_rows[table_name]) != expected:
        raise ValueError(f"Unexpected row count for {table_name}")

# Fixture integrity checks are plain Python and run before any table writes.
gpu_ids = {row["gpu_id"] for row in evidence_rows["inventory"]}
if len(gpu_ids) != 24:
    raise ValueError("Inventory must contain 24 unique GPUs")
if any(row["model"] != "Synthetic-" + row["vendor"] for row in evidence_rows["inventory"]):
    raise ValueError("Fixture models must be labelled synthetic")
for table_name, entity in (("telemetry", "gpu_id"), ("workloads", "gpu_id"), ("cooling", "cooling_zone_id")):
    rows = evidence_rows[table_name]
    keys = {(row["run_id"], row["event_ts"], row[entity]) for row in rows}
    if len(keys) != len(rows):
        raise ValueError(f"Duplicate temporal keys in {table_name}")
    if len({row["run_id"] for row in rows}) != 4:
        raise ValueError(f"Expected four opaque runs in {table_name}")
    if any(not row["event_ts"].endswith("Z") for row in rows):
        raise ValueError("Every timestamp must be explicitly UTC")
for row in evidence_rows["telemetry"]:
    if row["gpu_id"] not in gpu_ids:
        raise ValueError("Unknown GPU in telemetry")
    if row["heartbeat_ok"] == "false" and any(row[field] != "" for field in NULLABLE_METRICS):
        raise ValueError("Missing telemetry must contain empty metrics, not zeroes")

print("Synthetic CSV checks passed. Temporary evidence and evaluation files have been deleted.")


In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType,
    BooleanType, TimestampType,
)

double_columns = {
    "power_limit_w", "temperature_limit_c", "power_w", "gpu_util_pct",
    "memory_used_gb", "memory_total_gb", "temp_c", "clock_mhz",
    "inlet_temp_c", "flow_pct",
}
long_columns = {"ecc_uncorrected_total", "progress_units"}
boolean_columns = {"throttled", "heartbeat_ok"}

def field_type(name):
    if name == "event_ts":
        return TimestampType()
    if name in double_columns:
        return DoubleType()
    if name in long_columns:
        return LongType()
    if name in boolean_columns:
        return BooleanType()
    return StringType()

def convert_cell(table_name, name, value):
    if value == "":
        allowed = (table_name == "telemetry" and name in NULLABLE_METRICS) or (table_name == "workloads" and name == "job_id")
        if not allowed:
            raise ValueError(f"Unexpected empty value: {table_name}.{name}")
        return None
    if name == "event_ts":
        return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(timezone.utc)
    if name in double_columns:
        converted = float(value)
        if not math.isfinite(converted):
            raise ValueError(f"Non-finite number in {table_name}.{name}")
        return converted
    if name in long_columns:
        return int(value)
    if name in boolean_columns:
        if value not in ("true", "false"):
            raise ValueError(f"Invalid boolean in {table_name}.{name}")
        return value == "true"
    return value

frames = {}
for table_name, columns in SCHEMAS.items():
    # Nullable Spark fields match persisted Delta schemas. Required values
    # are checked during conversion; empty metrics are explicitly None.
    explicit_schema = StructType([StructField(name, field_type(name), True) for name in columns])
    converted_rows = [tuple(convert_cell(table_name, name, row[name]) for name in columns) for row in evidence_rows[table_name]]
    frames[table_name] = spark.createDataFrame(converted_rows, schema=explicit_schema)
print("Created explicitly typed Spark DataFrames; no schema inference was used.")


## Persist only evidence

The next cell checks **all existing target tables before writing**. Identical
schema and data are reused. A difference stops the load and asks for a new
schema name. New tables use SQL `CREATE TABLE ... USING DELTA AS SELECT`.
This avoids a DataFrame write mode that some environments do not support.


In [ ]:
qualified_names = {name: f"`{CATALOG}`.`{SCHEMA}`.`{name}`" for name in SCHEMAS}
already_loaded = set()
conflicts = []
for table_name, incoming in frames.items():
    qualified_name = qualified_names[table_name]
    if spark.catalog.tableExists(qualified_name):
        existing = spark.table(qualified_name)
        if existing.schema != incoming.schema:
            conflicts.append(f"{table_name}: schema differs")
            continue
        # exceptAll preserves duplicate counts. Check both directions.
        different = existing.exceptAll(incoming).limit(1).count() > 0 or incoming.exceptAll(existing).limit(1).count() > 0
        if different:
            conflicts.append(f"{table_name}: data differs")
        else:
            already_loaded.add(table_name)

if conflicts:
    raise ValueError("Existing tables were not changed. Choose a NEW SCHEMA name in the configuration cell and run all again. Conflicts: " + "; ".join(conflicts))

from uuid import uuid4

for table_name, incoming in frames.items():
    if table_name in already_loaded:
        print(f"Unchanged: {qualified_names[table_name]}")
    else:
        temp_view = f"_gpu_ops_load_{uuid4().hex}"
        incoming.createOrReplaceTempView(temp_view)
        try:
            # CREATE fails if another writer creates the target after preflight.
            # Collect finishes the command before its source view is removed.
            spark.sql(
                f"CREATE TABLE {qualified_names[table_name]} "
                f"USING DELTA AS SELECT * FROM `{temp_view}`"
            ).collect()
        finally:
            spark.catalog.dropTempView(temp_view)
        print(f"Created: {qualified_names[table_name]}")

counts = []
for table_name, expected in expected_counts.items():
    actual = spark.table(qualified_names[table_name]).count()
    if actual != expected:
        raise ValueError(f"Stored row count mismatch in {table_name}: {actual}, expected {expected}")
    counts.append((table_name, actual))
display(spark.createDataFrame(counts, "table_name STRING, row_count LONG"))
display(spark.table(qualified_names["inventory"]).orderBy("gpu_id"))


## Example queries for the website and agent

These are the exact SQL files shipped in the starter. Parameters are bound
through `spark.sql(..., args=...)`; they are not pasted into SQL strings.
Every temporal source is filtered by an opaque run ID and the replay cursor.
Unknown or stale telemetry remains distinguishable from a healthy reading.

The initial run is the first opaque ID; it is not chosen using its answer
label. Different runs can represent different conditions. Advance or move
the cursor between `09:00:00Z` and `09:14:55Z`, or select a different ID from
the displayed list. Queries do not create diagnoses or model outputs.


In [ ]:
QUERIES = {'fleet_evidence': "-- Bounded evidence for the canonical deterministic flagger adapter.\n-- Fetch one extra row per stream so the backend rejects overflow, never partial evidence.\n-- JSON preserves SQL nulls; timestamps are serialized in UTC. No evaluation labels are read.\nWITH inventory_bounded AS (\n  SELECT gpu_id, vendor, model, rack_id, server_id, cooling_zone_id, power_limit_w, temperature_limit_c FROM inventory\n  LIMIT 25\n),\ntelemetry_bounded AS (\n  SELECT run_id, event_ts, gpu_id, power_w, gpu_util_pct, memory_used_gb, memory_total_gb, temp_c, clock_mhz, throttled, ecc_uncorrected_total, heartbeat_ok FROM telemetry\n  WHERE run_id = :run_id AND event_ts >= :window_start_ts AND event_ts <= :as_of_ts\n  LIMIT 4345\n),\nworkloads_bounded AS (\n  SELECT run_id, event_ts, gpu_id, job_id, job_state, progress_units, priority FROM workloads\n  WHERE run_id = :run_id AND event_ts >= :window_start_ts AND event_ts <= :as_of_ts\n  LIMIT 4345\n),\ncooling_bounded AS (\n  SELECT run_id, event_ts, cooling_zone_id, inlet_temp_c, flow_pct FROM cooling\n  WHERE run_id = :run_id AND event_ts >= :window_start_ts AND event_ts <= :as_of_ts\n  LIMIT 4345\n)\nSELECT 'inventory' AS dataset, COUNT(*) AS row_count,\n  to_json(collect_list(named_struct('gpu_id', gpu_id, 'vendor', vendor, 'model', model, 'rack_id', rack_id, 'server_id', server_id, 'cooling_zone_id', cooling_zone_id, 'power_limit_w', power_limit_w, 'temperature_limit_c', temperature_limit_c)), map('ignoreNullFields', 'false', 'timeZone', 'UTC')) AS payload\nFROM inventory_bounded\nUNION ALL\nSELECT 'telemetry' AS dataset, COUNT(*) AS row_count,\n  to_json(collect_list(named_struct('run_id', run_id, 'event_ts', event_ts, 'gpu_id', gpu_id, 'power_w', power_w, 'gpu_util_pct', gpu_util_pct, 'memory_used_gb', memory_used_gb, 'memory_total_gb', memory_total_gb, 'temp_c', temp_c, 'clock_mhz', clock_mhz, 'throttled', throttled, 'ecc_uncorrected_total', ecc_uncorrected_total, 'heartbeat_ok', heartbeat_ok)), map('ignoreNullFields', 'false', 'timeZone', 'UTC')) AS payload\nFROM telemetry_bounded\nUNION ALL\nSELECT 'workloads' AS dataset, COUNT(*) AS row_count,\n  to_json(collect_list(named_struct('run_id', run_id, 'event_ts', event_ts, 'gpu_id', gpu_id, 'job_id', job_id, 'job_state', job_state, 'progress_units', progress_units, 'priority', priority)), map('ignoreNullFields', 'false', 'timeZone', 'UTC')) AS payload\nFROM workloads_bounded\nUNION ALL\nSELECT 'cooling' AS dataset, COUNT(*) AS row_count,\n  to_json(collect_list(named_struct('run_id', run_id, 'event_ts', event_ts, 'cooling_zone_id', cooling_zone_id, 'inlet_temp_c', inlet_temp_c, 'flow_pct', flow_pct)), map('ignoreNullFields', 'false', 'timeZone', 'UTC')) AS payload\nFROM cooling_bounded;\n", 'gpu_history': '-- Bind run_id/gpu_id (STRING), window_start_ts/as_of_ts (TIMESTAMP).\n-- Newest first; reverse rows for a chronological chart. At most 180 samples.\n-- Null readings remain null. A missing heartbeat is not a zero-power GPU.\nSELECT run_id, event_ts, gpu_id, power_w, gpu_util_pct,\n       memory_used_gb, memory_total_gb, temp_c, clock_mhz,\n       throttled, ecc_uncorrected_total, heartbeat_ok\nFROM telemetry\nWHERE run_id = :run_id\n  AND gpu_id = :gpu_id\n  AND event_ts >= :window_start_ts\n  AND event_ts <= :as_of_ts\nORDER BY event_ts DESC\nLIMIT 180;\n', 'health_alerts': "-- Bind run_id (STRING), as_of_ts (TIMESTAMP).\n-- Exact evaluated snapshot only: an empty result does not mean all GPUs are healthy.\n-- Include clear/unknown evaluations so the UI can distinguish them from active alerts.\nSELECT alert_id, run_id, as_of_ts, gpu_id, rack_id, vendor,\n       rule_id, rule_version, status, severity, summary, evidence_json\nFROM health_alerts\nWHERE run_id = :run_id\n  AND as_of_ts = :as_of_ts\n  AND rule_version = 'baseline-v1'\nORDER BY gpu_id, rule_id\nLIMIT 1000;\n", 'health_features': "-- Databricks processing step, not a diagnosis. All reads stop at the replay cursor.\n-- Bind run_id STRING, as_of_ts TIMESTAMP, window_start_ts TIMESTAMP, rule_version STRING.\nWITH t_latest AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY gpu_id ORDER BY event_ts DESC) AS rn\n  FROM telemetry WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), t_window AS (\n  SELECT *,\n    LAG(ecc_uncorrected_total) OVER (PARTITION BY gpu_id ORDER BY event_ts) AS prior_ecc,\n    LAG(event_ts) OVER (PARTITION BY gpu_id ORDER BY event_ts) AS prior_ts\n  FROM telemetry\n  WHERE run_id = :run_id AND event_ts >= :window_start_ts AND event_ts <= :as_of_ts\n), t_stats AS (\n  SELECT gpu_id,\n    unix_timestamp(MAX(event_ts)) - unix_timestamp(MIN(event_ts)) AS window_seconds,\n    MAX(ecc_uncorrected_total) - MIN(ecc_uncorrected_total) AS counter_delta,\n    COUNT(*) AS samples, COUNT(ecc_uncorrected_total) AS counter_samples,\n    SUM(CASE WHEN heartbeat_ok = TRUE THEN 1 ELSE 0 END) AS observed_samples,\n    MAX(CASE WHEN ecc_uncorrected_total < prior_ecc THEN 1 ELSE 0 END) AS counter_reset,\n    MAX(unix_timestamp(event_ts) - unix_timestamp(prior_ts)) AS max_gap\n  FROM t_window GROUP BY gpu_id\n), w_latest AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY gpu_id ORDER BY event_ts DESC) AS rn\n  FROM workloads WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), w_window AS (\n  SELECT *,\n    LAG(progress_units) OVER (PARTITION BY gpu_id, job_id ORDER BY event_ts) AS prior_progress,\n    LAG(event_ts) OVER (PARTITION BY gpu_id, job_id ORDER BY event_ts) AS prior_ts\n  FROM workloads\n  WHERE run_id = :run_id AND event_ts >= :window_start_ts AND event_ts <= :as_of_ts\n), w_stats AS (\n  SELECT gpu_id, job_id,\n    unix_timestamp(MAX(event_ts)) - unix_timestamp(MIN(event_ts)) AS window_seconds,\n    MAX(progress_units) - MIN(progress_units) AS progress_delta,\n    COUNT(*) AS samples, COUNT(progress_units) AS progress_samples,\n    SUM(CASE WHEN job_state = 'running' THEN 1 ELSE 0 END) AS running_samples,\n    MAX(CASE WHEN progress_units < prior_progress THEN 1 ELSE 0 END) AS counter_reset,\n    MAX(unix_timestamp(event_ts) - unix_timestamp(prior_ts)) AS max_gap\n  FROM w_window GROUP BY gpu_id, job_id\n), c_latest AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY cooling_zone_id ORDER BY event_ts DESC) AS rn\n  FROM cooling WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), joined AS (\n  SELECT i.*, t.event_ts AS telemetry_ts, t.heartbeat_ok,\n    t.power_w AS raw_power_w, t.gpu_util_pct AS raw_util, t.temp_c AS raw_temp_c,\n    t.throttled AS raw_throttled,\n    CASE WHEN t.heartbeat_ok = TRUE\n      AND unix_micros(:as_of_ts) - unix_micros(t.event_ts) <= 15000000\n      THEN TRUE ELSE FALSE END AS telemetry_is_observed,\n    ts.window_seconds AS ecc_window_seconds, ts.counter_delta,\n    CASE WHEN ts.samples >= 2 AND ts.samples = ts.counter_samples\n      AND ts.samples = ts.observed_samples AND ts.counter_reset = 0\n      AND ts.window_seconds >= 55 AND ts.max_gap <= 15\n      THEN TRUE ELSE FALSE END AS ecc_window_complete,\n    w.event_ts AS workload_ts, w.job_id, w.job_state,\n    CASE WHEN w.event_ts IS NOT NULL\n      AND unix_micros(:as_of_ts) - unix_micros(w.event_ts) <= 15000000\n      THEN TRUE ELSE FALSE END AS workload_is_fresh,\n    ws.window_seconds AS progress_window_seconds, ws.progress_delta,\n    CASE WHEN ws.samples >= 2 AND ws.samples = ws.progress_samples\n      AND ws.samples = ws.running_samples AND ws.counter_reset = 0\n      AND ws.window_seconds >= 55 AND ws.max_gap <= 15\n      THEN TRUE ELSE FALSE END AS progress_window_complete,\n    c.event_ts AS cooling_ts, c.inlet_temp_c AS raw_inlet_temp_c, c.flow_pct AS raw_flow_pct,\n    CASE WHEN c.event_ts IS NOT NULL\n      AND unix_micros(:as_of_ts) - unix_micros(c.event_ts) <= 15000000\n      THEN TRUE ELSE FALSE END AS cooling_is_fresh\n  FROM inventory i\n  LEFT JOIN t_latest t ON i.gpu_id = t.gpu_id AND t.rn = 1\n  LEFT JOIN t_stats ts ON i.gpu_id = ts.gpu_id\n  LEFT JOIN w_latest w ON i.gpu_id = w.gpu_id AND w.rn = 1\n  LEFT JOIN w_stats ws ON i.gpu_id = ws.gpu_id AND w.job_id = ws.job_id\n  LEFT JOIN c_latest c ON i.cooling_zone_id = c.cooling_zone_id AND c.rn = 1\n)\nSELECT :run_id AS run_id, :as_of_ts AS as_of_ts, :rule_version AS rule_version,\n  gpu_id, rack_id, vendor, cooling_zone_id,\n  -- Preserve the existing BIGINT display column; freshness above uses exact microseconds.\n  telemetry_ts, unix_timestamp(:as_of_ts) - unix_timestamp(telemetry_ts) AS telemetry_age_seconds,\n  telemetry_is_observed,\n  CASE WHEN telemetry_is_observed THEN raw_power_w END AS power_w,\n  CASE WHEN telemetry_is_observed THEN raw_util END AS gpu_util_pct,\n  CASE WHEN telemetry_is_observed THEN raw_temp_c END AS temp_c,\n  CASE WHEN telemetry_is_observed THEN raw_throttled END AS throttled,\n  power_limit_w, temperature_limit_c,\n  CASE WHEN telemetry_is_observed AND power_limit_w > 0 THEN raw_power_w / power_limit_w END AS power_ratio,\n  CASE WHEN telemetry_is_observed THEN temperature_limit_c - raw_temp_c END AS temperature_headroom_c,\n  CASE WHEN telemetry_is_observed AND ecc_window_complete THEN counter_delta END AS ecc_delta_60s,\n  ecc_window_seconds, workload_ts, workload_is_fresh, job_id, job_state,\n  CASE WHEN workload_is_fresh AND progress_window_complete THEN progress_delta END AS progress_delta_60s,\n  progress_window_seconds, cooling_ts, cooling_is_fresh,\n  CASE WHEN cooling_is_fresh THEN raw_inlet_temp_c END AS inlet_temp_c,\n  CASE WHEN cooling_is_fresh THEN raw_flow_pct END AS flow_pct\nFROM joined\nORDER BY gpu_id;\n", 'official_results': '-- Official detector audit records for one exact replay cursor; never baseline rules.\nSELECT run_id, as_of_ts, detector_version, evidence_sha256, incident_count,\n       snapshot_json, generated_at\nFROM official_detector_results\nWHERE run_id = :run_id AND as_of_ts = :as_of_ts\nORDER BY generated_at DESC\nLIMIT 50;\n', 'overview': '-- Bind run_id (STRING) and as_of_ts (TIMESTAMP). Set catalog/schema first.\n-- Synthetic replay: no sample later than the replay cursor is visible.\n-- Power is the sum of reporting GPUs, not rack/facility consumption.\nWITH ranked AS (\n  SELECT *,\n         ROW_NUMBER() OVER (PARTITION BY gpu_id ORDER BY event_ts DESC) AS sample_rank\n  FROM telemetry\n  WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), snapshot AS (\n  SELECT i.gpu_id, t.power_w, t.gpu_util_pct, t.temp_c,\n         CASE WHEN t.heartbeat_ok = TRUE\n                   AND unix_micros(:as_of_ts) - unix_micros(t.event_ts) <= 15000000\n              THEN TRUE ELSE FALSE END AS is_observed\n  FROM inventory AS i\n  LEFT JOIN ranked AS t ON i.gpu_id = t.gpu_id AND t.sample_rank = 1\n)\nSELECT COUNT(*) AS total_gpu_count,\n       COUNT(CASE WHEN is_observed THEN 1 END) AS observed_gpu_count,\n       COUNT(CASE WHEN NOT is_observed THEN 1 END) AS unknown_gpu_count,\n       COUNT(CASE WHEN is_observed AND power_w IS NOT NULL THEN 1 END)\n         AS power_reporting_gpu_count,\n       SUM(CASE WHEN is_observed THEN power_w END) / 1000.0\n         AS observed_gpu_power_kw,\n       AVG(CASE WHEN is_observed THEN gpu_util_pct END)\n         AS observed_avg_gpu_util_pct,\n       AVG(CASE WHEN is_observed THEN temp_c END)\n         AS observed_avg_temp_c\nFROM snapshot;\n', 'rack_context': "-- Bind run_id/rack_id (STRING), as_of_ts (TIMESTAMP).\n-- Returns raw last-known values with explicit freshness flags, not diagnoses.\n-- Consumers must check each stream's flag before interpreting a reading.\nWITH telemetry_ranked AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY gpu_id ORDER BY event_ts DESC) AS rn\n  FROM telemetry\n  WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), workload_ranked AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY gpu_id ORDER BY event_ts DESC) AS rn\n  FROM workloads\n  WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), cooling_ranked AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY cooling_zone_id ORDER BY event_ts DESC) AS rn\n  FROM cooling\n  WHERE run_id = :run_id AND event_ts <= :as_of_ts\n)\nSELECT i.gpu_id, i.vendor, i.model, i.rack_id, i.server_id,\n       i.cooling_zone_id, i.power_limit_w, i.temperature_limit_c,\n       t.event_ts AS telemetry_ts,\n       (unix_micros(:as_of_ts) - unix_micros(t.event_ts)) / 1000000.0 AS telemetry_age_seconds,\n       CASE WHEN t.event_ts IS NOT NULL\n                   AND unix_micros(:as_of_ts) - unix_micros(t.event_ts) <= 15000000\n              THEN TRUE ELSE FALSE END AS telemetry_is_fresh,\n       CASE WHEN t.heartbeat_ok = TRUE\n                   AND unix_micros(:as_of_ts) - unix_micros(t.event_ts) <= 15000000\n              THEN TRUE ELSE FALSE END AS telemetry_is_observed,\n       t.heartbeat_ok, t.power_w, t.gpu_util_pct, t.memory_used_gb,\n       t.memory_total_gb, t.temp_c, t.clock_mhz, t.throttled, t.ecc_uncorrected_total,\n       w.event_ts AS workload_ts,\n       CASE WHEN w.event_ts IS NOT NULL\n                   AND unix_micros(:as_of_ts) - unix_micros(w.event_ts) <= 15000000\n              THEN TRUE ELSE FALSE END AS workload_is_fresh,\n       w.job_id, w.job_state, w.progress_units, w.priority,\n       c.event_ts AS cooling_ts,\n       CASE WHEN c.event_ts IS NOT NULL\n                   AND unix_micros(:as_of_ts) - unix_micros(c.event_ts) <= 15000000\n              THEN TRUE ELSE FALSE END AS cooling_is_fresh,\n       c.inlet_temp_c, c.flow_pct\nFROM inventory AS i\nLEFT JOIN telemetry_ranked AS t ON i.gpu_id = t.gpu_id AND t.rn = 1\nLEFT JOIN workload_ranked AS w ON i.gpu_id = w.gpu_id AND w.rn = 1\nLEFT JOIN cooling_ranked AS c ON i.cooling_zone_id = c.cooling_zone_id AND c.rn = 1\nWHERE i.rack_id = :rack_id\nORDER BY i.gpu_id\nLIMIT 24;\n", 'runs': '-- Replay metadata only: no scenario names or ground-truth failure labels.\n-- Choose a returned run_id and use its time range as the replay cursor bounds.\nSELECT run_id,\n       MIN(event_ts) AS start_ts,\n       MAX(event_ts) AS end_ts,\n       COUNT(*) AS sample_count\nFROM telemetry\nGROUP BY run_id\nORDER BY run_id\nLIMIT 100;\n', 'workload_history': '-- Bind run_id/gpu_id (STRING), window_start_ts/as_of_ts (TIMESTAMP).\n-- Historical evidence for progress/stall detection; newest first.\nSELECT run_id, event_ts, gpu_id, job_id, job_state, progress_units, priority\nFROM workloads\nWHERE run_id = :run_id\n  AND gpu_id = :gpu_id\n  AND event_ts >= :window_start_ts\n  AND event_ts <= :as_of_ts\nORDER BY event_ts DESC\nLIMIT 180;\n'}


In [ ]:
runs = spark.table(qualified_names["telemetry"]).select("run_id").distinct().orderBy("run_id")
display(runs)
RUN_ID = runs.first()["run_id"]
AS_OF_TS = datetime(2026, 11, 14, 9, 7, 30, tzinfo=timezone.utc)
WINDOW_START_TS = AS_OF_TS - timedelta(minutes=5)
first_gpu = spark.table(qualified_names["inventory"]).orderBy("gpu_id").first()
GPU_ID = first_gpu["gpu_id"]
RACK_ID = first_gpu["rack_id"]

print(f"Replay: run={RUN_ID}, cursor={AS_OF_TS.isoformat()}, gpu={GPU_ID}, rack={RACK_ID}")
display(spark.sql(QUERIES["overview"], args={"run_id": RUN_ID, "as_of_ts": AS_OF_TS}))
display(spark.sql(QUERIES["gpu_history"], args={
    "run_id": RUN_ID, "gpu_id": GPU_ID,
    "window_start_ts": WINDOW_START_TS, "as_of_ts": AS_OF_TS,
}))
display(spark.sql(QUERIES["rack_context"], args={
    "run_id": RUN_ID, "rack_id": RACK_ID, "as_of_ts": AS_OF_TS,
}))


## Share with teammates

Share the four table names, the SQL files and their parameter names. The
website should display telemetry timestamps, use “unknown” for unavailable
readings, and label summed power as **GPU power**, not total facility power.

For the model teammates, agree on one incident-output format and evaluate
them on the same evidence windows. This notebook creates neither the
deterministic detector nor the fine-tuned model nor an autonomous repair loop.
Keep answer labels outside the agent's data access; use separate evaluation
data when measuring accuracy.

References: [Databricks serverless limits](https://docs.databricks.com/aws/en/compute/serverless/limitations),
[Spark SQL bound parameters](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql.SparkSession.sql.html),
[SQL Statement Execution API](https://docs.databricks.com/aws/en/dev-tools/sql-execution-tutorial).
